# EU27 — Target Achievement Predictive Model
### Time Series Forecast to 2030 | Cluster-informed Analysis

**Approach:** For each of the 27 Member States, fit a linear trend on 2005–2024 historical data  
and project to 2030. Cluster labels (reduced, k=4) are used to interpret results at group level.

**Three binding Fit-for-55 dimensions:**

| Dimension | Direction | 2030 Target |
|-----------|-----------|-------------|
| GHG emissions (`GHG_MtCO2eq`) | ↓ reduce | Country-specific (NECP) |
| Final Energy Consumption (`FEC_Mtoe`) | ↓ reduce | Country-specific (NECP) |
| Renewable Energy Share (`RES_Share_Pct`) | ↑ increase | Country-specific (NECP) |

**Inputs:** `EU27_ESG_Panel_2005_2024.csv`, `EU27_cluster_labels_reduced.csv`  
**Outputs:** `forecast_results.csv`, `forecast_trajectories.png`, `cluster_heatmap.png`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score, mean_absolute_error
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

# ── CONFIG ─────────────────────────────────────────────────────────────────────
PANEL_PATH   = "EU27_ESG_Panel_2005_2024.csv"
CLUSTER_PATH = "EU27_cluster_labels_reduced.csv"
FORECAST_YEAR = 2030
TRAIN_FROM    = 2005        # use full history for trend
POLY_DEGREE   = 1           # 1 = linear, 2 = quadratic — change to compare
CI_LEVEL      = 0.90        # confidence interval for forecast band

DARK_BG  = "#0f1117"
PANEL_BG = "#1a1d2e"
CLUSTER_COLORS = {1: "#ff6b6b", 2: "#00d4ff", 3: "#7fff7f", 4: "#ffd700"}
CLUSTER_NAMES  = {
    1: "Large Emitters",        # FR, IT, PL, ES
    2: "Heterogeneous Middle",  # 13 countries
    3: "Green Transition Leaders", # AT, DK, FI, LV, PT, SE
    4: "Germany (Singleton)",   # DE
}
# ──────────────────────────────────────────────────────────────────────────────
print("Config loaded ✓")


## 1. Load & Merge Data

In [ ]:
df = pd.read_csv(PANEL_PATH)
clusters = pd.read_csv(CLUSTER_PATH)

df = df.merge(clusters[["Country_Code", "Cluster_k4"]], on="Country_Code")
df = df[df["Year"] >= TRAIN_FROM].sort_values(["Country_Code", "Year"])

print(f"Panel: {df.shape[0]} rows × {df.shape[1]} cols")
print(f"Countries: {df['Country_Code'].nunique()}  |  Years: {sorted(df['Year'].unique())}")
print(f"\nCluster distribution:")
for c, name in CLUSTER_NAMES.items():
    members = df[df['Cluster_k4']==c]['Country_Code'].unique()
    print(f"  Cluster {c} — {name}: {', '.join(sorted(members))}")


## 2. Forecasting Function

For each country and each target dimension we:
1. Fit a polynomial trend (degree set in CONFIG — default linear) on the full 2005–2024 series
2. Project to 2030
3. Compute a prediction interval using residual standard error
4. Flag on-track / at-risk based on whether the projection crosses the NECP target


In [ ]:
TARGETS = {
    "GHG": {
        "actual":  "GHG_MtCO2eq",
        "target":  "GHG_FF55_Target_Mt",
        "label":   "GHG Emissions (MtCO₂eq)",
        "lower_is_better": True,
    },
    "FEC": {
        "actual":  "FEC_Mtoe",
        "target":  "FEC_FF55_Target_Mtoe",
        "label":   "Final Energy Consumption (Mtoe)",
        "lower_is_better": True,
    },
    "RES": {
        "actual":  "RES_Share_Pct",
        "target":  "RES_FF55_Target_Pct",
        "label":   "Renewable Energy Share (%)",
        "lower_is_better": False,
    },
}

def forecast_country(grp, dim_cfg, degree=POLY_DEGREE, ci=CI_LEVEL):
    """Fit polynomial trend, return projection + CI at FORECAST_YEAR."""
    years = grp["Year"].values.astype(float)
    y     = grp[dim_cfg["actual"]].values.astype(float)
    target_val = grp[dim_cfg["target"]].iloc[0]

    poly = PolynomialFeatures(degree=degree, include_bias=True)
    Xp   = poly.fit_transform(years.reshape(-1, 1))
    lr   = LinearRegression(fit_intercept=False).fit(Xp, y)

    # Residuals → prediction interval
    y_pred  = lr.predict(Xp)
    resid   = y - y_pred
    s       = np.std(resid, ddof=degree+1)
    n       = len(y)
    t_crit  = stats.t.ppf((1 + ci) / 2, df=n - degree - 1)
    margin  = t_crit * s * np.sqrt(1 + 1/n)

    # Project to 2030
    X2030   = poly.transform([[float(FORECAST_YEAR)]])
    proj    = lr.predict(X2030)[0]
    lo, hi  = proj - margin, proj + margin

    lib = dim_cfg["lower_is_better"]
    on_track = proj <= target_val if lib else proj >= target_val

    return {
        "proj":      round(proj, 3),
        "ci_lo":     round(lo, 3),
        "ci_hi":     round(hi, 3),
        "target":    round(target_val, 3),
        "gap":       round(proj - target_val, 3),
        "on_track":  on_track,
        "r2":        round(r2_score(y, y_pred), 3),
        "mae":       round(mean_absolute_error(y, y_pred), 3),
        "slope":     round(lr.coef_[1], 4),   # linear coefficient
    }

print("Forecast function defined ✓")


## 3. Run Forecasts for All 27 Countries

In [ ]:
records = []

for code, grp in df.groupby("Country_Code"):
    grp = grp.sort_values("Year")
    cluster  = grp["Cluster_k4"].iloc[0]
    country  = grp["Country_Name"].iloc[0]
    row = {"Country_Code": code, "Country": country, "Cluster": cluster,
           "Cluster_Name": CLUSTER_NAMES[cluster]}

    for dim, cfg in TARGETS.items():
        fc = forecast_country(grp, cfg)
        for k, v in fc.items():
            row[f"{dim}_{k}"] = v

    records.append(row)

results = pd.DataFrame(records).sort_values(["Cluster", "Country"])
print(f"Forecasts computed for {len(results)} countries\n")

# Summary columns for quick view
summary_cols = ["Country", "Cluster_Name",
                "GHG_proj", "GHG_target", "GHG_on_track",
                "FEC_proj", "FEC_target", "FEC_on_track",
                "RES_proj", "RES_target", "RES_on_track"]

def style_bool(v):
    if v is True:  return "background-color: #1a3d1a; color: #7fff7f"
    if v is False: return "background-color: #3d1a1a; color: #ff6b6b"
    return ""

results[summary_cols].style.applymap(
    style_bool, subset=["GHG_on_track","FEC_on_track","RES_on_track"]
).format(precision=2)


## 4. Cluster-Level Summary

In [ ]:
summary = results.groupby(["Cluster","Cluster_Name"]).agg(
    Countries      = ("Country_Code", "count"),
    GHG_on_track   = ("GHG_on_track", "sum"),
    FEC_on_track   = ("FEC_on_track", "sum"),
    RES_on_track   = ("RES_on_track", "sum"),
    GHG_avg_gap    = ("GHG_gap", "mean"),
    FEC_avg_gap    = ("FEC_gap", "mean"),
    RES_avg_gap    = ("RES_gap", "mean"),
).reset_index()

for dim in ["GHG","FEC","RES"]:
    summary[f"{dim}_pct_track"] = (summary[f"{dim}_on_track"] / summary["Countries"] * 100).round(0).astype(int).astype(str) + "%"

print(summary[["Cluster_Name","Countries",
               "GHG_on_track","GHG_pct_track","GHG_avg_gap",
               "FEC_on_track","FEC_pct_track","FEC_avg_gap",
               "RES_on_track","RES_pct_track","RES_avg_gap"]].to_string(index=False))


## 5. Trajectory Plots by Cluster

One panel per cluster, one line per country, for each of the three dimensions.  
Dashed horizontal line = 2030 target. Shaded band = 90% prediction interval.


In [ ]:
PROJ_YEARS = np.arange(2005, 2031)

fig, axes = plt.subplots(4, 3, figsize=(20, 22))
fig.patch.set_facecolor(DARK_BG)

for ci_idx, (cluster_id, cluster_name) in enumerate(CLUSTER_NAMES.items()):
    cluster_countries = df[df["Cluster_k4"] == cluster_id]["Country_Code"].unique()
    c_color = CLUSTER_COLORS[cluster_id]

    for di_idx, (dim, cfg) in enumerate(TARGETS.items()):
        ax = axes[ci_idx][di_idx]
        ax.set_facecolor(PANEL_BG)

        for code in sorted(cluster_countries):
            grp = df[df["Country_Code"] == code].sort_values("Year")
            years_hist = grp["Year"].values.astype(float)
            y_hist     = grp[cfg["actual"]].values.astype(float)
            target_val = grp[cfg["target"]].iloc[0]

            # Refit for plotting
            poly = PolynomialFeatures(POLY_DEGREE, include_bias=True)
            Xp   = poly.fit_transform(years_hist.reshape(-1,1))
            lr   = LinearRegression(fit_intercept=False).fit(Xp, y_hist)
            y_pred_proj = lr.predict(poly.transform(PROJ_YEARS.reshape(-1,1)))

            # CI band
            resid = y_hist - lr.predict(Xp)
            s     = np.std(resid, ddof=POLY_DEGREE+1)
            n     = len(y_hist)
            t_crit = stats.t.ppf((1 + CI_LEVEL) / 2, df=n - POLY_DEGREE - 1)
            margin = t_crit * s * np.sqrt(1 + 1/n)

            on_track = results.loc[results["Country_Code"]==code, f"{dim}_on_track"].values[0]
            line_color = "#7fff7f" if on_track else "#ff6b6b"
            alpha_hist = 0.9

            ax.plot(years_hist, y_hist, color=c_color, linewidth=1.2, alpha=0.5)
            ax.plot(PROJ_YEARS, y_pred_proj, color=line_color, linewidth=1.5,
                    linestyle="--", alpha=0.8)
            ax.fill_between(PROJ_YEARS,
                            y_pred_proj - margin, y_pred_proj + margin,
                            color=line_color, alpha=0.04)

            # Label endpoint
            ax.annotate(code,
                        xy=(2030, y_pred_proj[-1]),
                        fontsize=6, color=line_color, alpha=0.85,
                        xytext=(2, 0), textcoords="offset points")

        # Target line
        target_vals = df[(df["Cluster_k4"]==cluster_id)][cfg["target"]].unique()
        for tv in target_vals:
            ax.axhline(tv, color="#ffd700", linewidth=0.6, alpha=0.3)

        ax.axvline(2024, color="white", linewidth=0.8, alpha=0.3, linestyle=":")
        ax.axvline(2030, color="#ffd700", linewidth=0.8, alpha=0.4, linestyle=":")

        if ci_idx == 0:
            ax.set_title(cfg["label"], color="white", fontsize=10, fontweight="bold", pad=8)
        if di_idx == 0:
            ax.set_ylabel(f"Cluster {cluster_id}\n{cluster_name}", color=c_color,
                          fontsize=8, fontweight="bold")

        ax.tick_params(colors="#aaa", labelsize=7)
        ax.spines[:].set_edgecolor("#333")

        # Legend
        from matplotlib.lines import Line2D
        legend_els = [
            Line2D([0],[0], color="#7fff7f", linestyle="--", label="On track"),
            Line2D([0],[0], color="#ff6b6b", linestyle="--", label="At risk"),
            Line2D([0],[0], color="#ffd700", linewidth=0.8, label="2030 target"),
        ]
        ax.legend(handles=legend_els, fontsize=6, facecolor=PANEL_BG,
                  labelcolor="white", loc="upper right", framealpha=0.6)

plt.suptitle("EU27 Target Achievement Trajectories to 2030\n(dashed = projection, solid = historical)",
             color="white", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("forecast_trajectories.png", dpi=150, bbox_inches="tight", facecolor=DARK_BG)
plt.show()
print("Saved → forecast_trajectories.png")


## 6. Country-Level Heatmap — Gap to 2030 Target

In [ ]:
import matplotlib.colors as mcolors

fig, axes = plt.subplots(1, 3, figsize=(20, 10))
fig.patch.set_facecolor(DARK_BG)

results_sorted = results.sort_values(["Cluster","Country"])

dims_heatmap = [
    ("GHG_gap", "GHG Gap to Target\n(MtCO₂eq, + = over target)", True),
    ("FEC_gap", "FEC Gap to Target\n(Mtoe, + = over target)",    True),
    ("RES_gap", "RES Gap to Target\n(pp, + = under target)",     False),
]

countries_ordered = results_sorted["Country"].values
cluster_labels    = results_sorted["Cluster"].values

for ax, (col, title, lower_bad) in zip(axes, dims_heatmap):
    ax.set_facecolor(PANEL_BG)
    vals = results_sorted[col].values.astype(float).reshape(-1, 1)

    # Diverging colormap: green = good, red = bad
    vmax = np.abs(vals).max()
    if lower_bad:
        norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
        cmap = "RdYlGn_r"
    else:
        norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
        cmap = "RdYlGn"

    im = ax.imshow(vals, aspect="auto", cmap=cmap, norm=norm)

    ax.set_xticks([])
    ax.set_yticks(range(len(countries_ordered)))
    ax.set_yticklabels(countries_ordered, fontsize=8, color="white")

    # Cluster colour stripe on left
    for i, (c_label) in enumerate(cluster_labels):
        ax.add_patch(plt.Rectangle((-1.5, i-0.5), 1, 1,
                                   color=CLUSTER_COLORS[c_label], clip_on=False))

    # Value annotations
    for i, v in enumerate(vals.flatten()):
        ax.text(0, i, f"{v:+.1f}", ha="center", va="center",
                fontsize=7, color="white", fontweight="bold")

    ax.set_title(title, color="white", fontsize=10, fontweight="bold", pad=10)
    cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.04)
    cbar.ax.tick_params(colors="#aaa", labelsize=7)

# Cluster legend
from matplotlib.patches import Patch
legend_patches = [Patch(color=CLUSTER_COLORS[c], label=f"Cluster {c}: {CLUSTER_NAMES[c]}")
                  for c in CLUSTER_NAMES]
fig.legend(handles=legend_patches, loc="lower center", ncol=2,
           facecolor=PANEL_BG, labelcolor="white", fontsize=9,
           bbox_to_anchor=(0.5, -0.03))

plt.suptitle("Projected 2030 Gap to Fit-for-55 Targets (Linear Trend Forecast)",
             color="white", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("cluster_heatmap.png", dpi=150, bbox_inches="tight", facecolor=DARK_BG)
plt.show()
print("Saved → cluster_heatmap.png")


## 7. Model Diagnostics

R² and MAE per country per dimension — flags where linear trend fits poorly  
(low R² = noisy or non-linear series, less reliable projection).


In [ ]:
diag_cols = ["Country","Cluster_Name",
             "GHG_r2","GHG_mae","FEC_r2","FEC_mae","RES_r2","RES_mae"]

diag = results[diag_cols].copy()

def flag_r2(v):
    if v < 0.5: return "background-color: #3d1a1a; color: #ff9999"
    if v < 0.75: return "background-color: #3d3000; color: #ffd700"
    return ""

diag.style.applymap(flag_r2, subset=["GHG_r2","FEC_r2","RES_r2"]).format(precision=3)


## 8. Export Results

In [ ]:
export_cols = [
    "Country_Code","Country","Cluster","Cluster_Name",
    "GHG_proj","GHG_ci_lo","GHG_ci_hi","GHG_target","GHG_gap","GHG_on_track","GHG_r2",
    "FEC_proj","FEC_ci_lo","FEC_ci_hi","FEC_target","FEC_gap","FEC_on_track","FEC_r2",
    "RES_proj","RES_ci_lo","RES_ci_hi","RES_target","RES_gap","RES_on_track","RES_r2",
]
results[export_cols].to_csv("forecast_results.csv", index=False)
print("Saved → forecast_results.csv")
results[export_cols]
